# Segment the cloth part of pool table

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
DATA_DIR = Path("./../data")

IMAGE_PATH = DATA_DIR / "development_set/4a_png.rf.a6bb5c5706fd8628eb53d34a122cf441.jpg"

In [ ]:
image = cv2.imread(str(IMAGE_PATH))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Display the image
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
image = cv2.GaussianBlur(image, (15, 15), 0)
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

# Plot all three channels of the HSV image
fig, axes = plt.subplots(1, 3, figsize=(120, 50))
axes[0].imshow(hsv_image[:, :, 0], cmap='hsv')
axes[0].set_title('H')
axes[0].axis('off')
axes[1].imshow(hsv_image[:, :, 1], cmap='gray')
axes[1].set_title('S')
axes[1].axis('off')
axes[2].imshow(hsv_image[:, :, 2], cmap='gray')
axes[2].set_title('V')
axes[2].axis('off')
plt.show()

In [ ]:
def get_upper_lower_blue(hsv_image) -> tuple[np.ndarray, np.ndarray, tuple[int, int, int, int]]:
    h, w = image.shape[:2]

    x0 = int(w * 0.3)
    x1 = int(w * 0.7)
    y0 = int(h * 0.3)
    y1 = int(h * 0.7)

    roi_hsv = hsv_image[y0:y1, x0:x1]

    H = roi_hsv[:, :, 0]
    S = roi_hsv[:, :, 1]
    V = roi_hsv[:, :, 2]

    # Vamos deixar fora os mais "gray" e "black" pixels
    valid = (S > 30) & (V > 30)

    hue_hist = np.bincount(H[valid].ravel(), minlength=180)
    dominant_hue = np.argmax(hue_hist)
    print(f"{dominant_hue=}")

    hue_window = 3
    near_hue = valid & (np.abs(H.astype(np.int16) - int(dominant_hue)) <= hue_window)

    s_low  = int(np.percentile(S[near_hue], 10))    # ignores the lowest 10% of pixels
    s_high = int(np.percentile(S[near_hue], 95))

    v_low  = int(np.percentile(V[near_hue], 5))
    v_high = int(np.percentile(V[near_hue], 95))

    lower_blue = np.array([
        max(0, dominant_hue - hue_window),
        max(0, s_low - 10),
        max(0, v_low - 10)
    ], dtype=np.uint8)

    upper_blue = np.array([
        min(179, dominant_hue + hue_window),
        min(255, s_high + 10),
        min(255, v_high + 10)
    ], dtype=np.uint8)

    return lower_blue, upper_blue, (x0, y0, x1, y1)

lower_blue, upper_blue, (x0, y0, x1, y1) = get_upper_lower_blue(hsv_image)

print(f"{lower_blue=}")
print(f"{upper_blue=}")

image_roi = image.copy()
cv2.rectangle(image_roi, (x0, y0), (x1, y1), (255, 255, 255), 10)
plt.imshow(image_roi)
plt.axis('off')
plt.show()

In [ ]:
# lower_blue = np.array([200/2, 60 / 100 * 255, 60 / 100 * 255])
# upper_blue = np.array([210/2, 90 / 100 * 255, 100 / 100 * 255])

mask_table = cv2.inRange(hsv_image, lower_blue, upper_blue)

# TODO: esperimentar com e sem isso:
# https://www.geeksforgeeks.org/python/python-opencv-morphological-operations/
kernel = np.ones((7, 7), np.uint8)
mask_table = cv2.morphologyEx(mask_table, cv2.MORPH_OPEN, kernel)
mask_table = cv2.morphologyEx(mask_table, cv2.MORPH_CLOSE, kernel)

plt.imshow(mask_table, cmap='gray')
plt.axis('off')
plt.title('Mask for Blue Color')
plt.show()

In [ ]:
# make 3 plots for each channel of RGD image
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(image[:, :, 0], cmap='gray')
axes[0].set_title('R')
axes[0].axis('off')
axes[1].imshow(image[:, :, 1], cmap='gray')
axes[1].set_title('G')
axes[1].axis('off')
axes[2].imshow(image[:, :, 2], cmap='gray')
axes[2].set_title('B')
axes[2].axis('off')
plt.show()

In [ ]:
# make histogram for each channel of the RGB image
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].hist(image[:, :, 0].ravel(), bins=50, color='red')
axes[0].set_title('R Histogram')
axes[1].hist(image[:, :, 1].ravel(), bins=50, color='green')
axes[1].set_title('G Histogram')
axes[2].hist(image[:, :, 2].ravel(), bins=50, color='blue')
axes[2].set_title('B Histogram')
plt.show()

In [ ]:
lower_blue = 200
upper_blue = 240

mask_table = cv2.inRange(image[:, :, 2], lower_blue, upper_blue)

plt.imshow(mask_table, cmap='gray')
plt.axis('off')
plt.title('Mask for Blue Color in B Channel')
plt.show()